# 06 · Build and audit a custom detection dataset

**Level:** intermediate · **Class time:** 45–60 minutes · **Prerequisites:** 01 and 03.
**Resources:** offline, CPU, no paid service or annotation account.

You will create 144 labeled images, understand YOLO annotation files, inspect labels,
check class balance and duplicates, and practice a manual annotation. **Presenter
question:** “Can a model learn an object that the annotator forgot to label?”


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the ObjectDetection project folder (see README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course_utils import seed_everything, show_boxes
seed_everything(42)
print("Project:", ROOT)


## Step 1 — Generate train, validation and test partitions

Our three classes are `rectangle=0`, `circle=1`, `triangle=2`. The generator uses
separate seeds for each split: 96 training, 24 validation and 24 test images.
Every twelfth image is intentionally empty. This helps distinguish true negatives
from missing annotations. Rerunning this cell reuses existing files.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from course_utils import (ensure_synthetic_dataset, audit_dataset, read_yolo_labels,
                          xyxy_to_yolo, CLASS_NAMES)
DATA_YAML = ensure_synthetic_dataset()
DATA_ROOT = DATA_YAML.parent
print(DATA_YAML.read_text())
print((DATA_ROOT / "manifest.json").read_text())


## Step 2 — Connect the files to the task

```
synthetic_shapes/
  data.yaml
  images/train/train_0001.png
  labels/train/train_0001.txt
  images/val/...
  labels/val/...
  images/test/...
  labels/test/...
```

For every object, a row holds `class_id center_x center_y width height`.
Coordinates are divided by image width or height and lie between zero and one.
Class IDs start at zero. The course writes explicit empty text files for negatives;
Ultralytics also permits missing label files for images with no objects.


In [ ]:
image_path = DATA_ROOT / "images/train/train_0001.png"
label_path = DATA_ROOT / "labels/train/train_0001.txt"
image = Image.open(image_path).convert("RGB")
print("Raw label file:\n", label_path.read_text())
boxes, labels = read_yolo_labels(label_path, *image.size)
show_boxes(image, boxes, [CLASS_NAMES[int(c)] for c in labels], title=image_path.name)
plt.show()


## Step 3 — Work through one annotation by hand

For a 256×256 image and `[32, 64, 128, 160]`, the center is `(80,112)`;
the width and height are both 96. Normalize x quantities by 256 and y quantities by
256. For a non-square image, those denominators differ.


In [ ]:
manual_box = np.array([[32., 64., 128., 160.]])
normalized = xyxy_to_yolo(manual_box, width=256, height=256)
print("class cx cy width height:", 0, *normalized[0])
assert np.allclose(normalized, [[0.3125, 0.4375, 0.375, 0.375]])


## Step 4 — See a grid before trusting a statistic

Look for boxes that are too loose, shifted coordinates, missing objects, incorrect
class IDs and truncated objects. The first image is a deliberately negative example.
We inspect training and validation images; keep the test split for the final project.


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, path in zip(axes.flat, sorted((DATA_ROOT / "images/train").glob("*.png"))[:8]):
    with Image.open(path) as im:
        b, c = read_yolo_labels(DATA_ROOT / "labels/train" / (path.stem + ".txt"), *im.size)
        show_boxes(im, b, [CLASS_NAMES[int(k)] for k in c], ax=ax, title=path.stem)
plt.tight_layout()
plt.show()


## Step 5 — Audit the actual files

The audit checks image readability, matching label files, finite normalized values,
positive box sizes, valid class IDs, in-bounds boxes and exact image duplicates across
splits. Hash checks cannot detect different frames of the same scene or near-duplicates;
split by recording session, object, location or person when collecting real data.


In [ ]:
audit = pd.DataFrame(audit_dataset(DATA_YAML))
display(audit)
audit.set_index("split")[CLASS_NAMES].plot.bar(figsize=(8, 4), title="Object counts, not image counts")
plt.ylabel("Annotated instances")
plt.tight_layout()
plt.show()


## Step 6 — Split by group before augmentation

Here twelve imaginary recording sessions each contribute five images. Shuffle the
session IDs, then assign all images of a session to one split. Do this **before**
augmentation. A rotated copy of a training image in validation is data leakage.
For real data, ensure every important class and condition has enough independent groups.


In [ ]:
records = pd.DataFrame([{"file": f"session{s:02d}_frame{f:03d}.jpg", "group": s}
                        for s in range(12) for f in range(5)])
rng = np.random.default_rng(42)
groups = rng.permutation(records["group"].unique())
assignment = {int(g): "train" if i < 8 else "val" if i < 10 else "test"
              for i, g in enumerate(groups)}
records["split"] = records["group"].map(assignment)
assert records.groupby("group")["split"].nunique().max() == 1
display(records.groupby("split").agg(images=("file", "size"), independent_groups=("group", "nunique")))


## Step 7 — Annotate a sample yourself

Choose a training image with one or more shapes. Display it without boxes and read its
pixel coordinates from the axes. Enter one box and a class ID for a visible object.
This exercise saves a separate annotation under `artifacts/annotation_practice/`;
it does not replace the generated training labels. A complete image annotation must
include **all** in-scope visible objects, not just the easiest one.


In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.title("Read pixel coordinates from the axes")
plt.grid(alpha=0.3)
plt.show()
print("Image dimensions:", image.size)


In [ ]:
# Start with one known box; move its edges to see how annotation quality changes.
practice_box = boxes[:1].copy()
practice_class = int(labels[0])
practice_dir = ROOT / "artifacts" / "annotation_practice"
practice_dir.mkdir(parents=True, exist_ok=True)
practice_path = practice_dir / "one_object.txt"
row = xyxy_to_yolo(practice_box, *image.size)[0]
practice_path.write_text(f"{practice_class} " + " ".join(f"{v:.8f}" for v in row) + "\n")
roundtrip_boxes, roundtrip_classes = read_yolo_labels(practice_path, *image.size)
show_boxes(image, roundtrip_boxes, [CLASS_NAMES[int(c)] for c in roundtrip_classes], title="Your annotation")
plt.show()
assert np.allclose(roundtrip_boxes, practice_box, atol=1e-4)


## Exercise and solution

**Exercise:** Explain what is wrong with `1 0.4 0.4 -0.2 0.3` and
`7 0.4 0.4 0.2 0.3`. How should an empty image be represented? Why does using a
random split of neighboring video frames inflate your confidence in a model?

**Solution:** the first has negative width; the second refers to an undefined class.
An empty label file represents a negative under this course's explicit convention.
Neighboring frames are correlated; the validation set becomes too similar to training.

**Real-data handoff:** replace the generator with your images and YOLO labels, define
consistent annotation rules (occlusion, tiny objects and class boundaries), group-split
the data, visually inspect labels, and point notebook 07's `DATA_YAML` to your config.
Do not train on confidential or unauthorized images.

**Next:** [07 — Train your detector](07_Train_a_Custom_YOLO_Detector.ipynb).
Reference: [Ultralytics detection dataset format](https://docs.ultralytics.com/datasets/detect/).
